In [1]:
# =========================
# 1. MOUNT GOOGLE DRIVE
# =========================
from google.colab import drive
drive.mount('/content/drive')


# =========================
# 2. IMPORTS
# =========================
import os
import copy
import pandas as pd
from PIL import Image
import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected. In Colab, go to Runtime > Change runtime type > GPU.")


# =========================
# 3. PATHS
# =========================
# CHANGE THESE TWO PATHS TO MATCH YOUR DRIVE
CSV_FILE = "/content/drive/MyDrive/ISIC/metadata.csv"
IMAGE_DIR = "/content/drive/MyDrive/ISIC/New folder"

MODEL_SAVE_PATH = "/content/drive/MyDrive/ISIC/models/densenet_skin_cancer_colab.pth"

os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)


# =========================
# 4. DATASET
# =========================
class ISICBinaryDataset(Dataset):
    def __init__(self, csv_file, image_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform

        image_col = "isic_id"
        label_col = "diagnosis_1"

        if image_col not in self.df.columns:
            raise ValueError(f"Column '{image_col}' not found in CSV.")

        if label_col not in self.df.columns:
            raise ValueError(f"Column '{label_col}' not found in CSV.")

        self.df[label_col] = self.df[label_col].astype(str).str.strip().str.lower()

        cancer_labels = {"malignant"}
        non_cancer_labels = {"benign"}

        self.df = self.df[
            self.df[label_col].isin(cancer_labels.union(non_cancer_labels))
        ].copy()

        self.df["target"] = self.df[label_col].apply(
            lambda x: 1 if x in cancer_labels else 0
        )

        self.image_col = image_col
        self.label_col = label_col

        print("Rows after filtering:", len(self.df))
        print("Class counts:")
        print(self.df[self.label_col].value_counts())

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_name = str(row[self.image_col])

        possible_paths = [
            os.path.join(self.image_dir, image_name),
            os.path.join(self.image_dir, image_name + ".jpg"),
            os.path.join(self.image_dir, image_name + ".jpeg"),
            os.path.join(self.image_dir, image_name + ".png"),
            os.path.join(self.image_dir, image_name + ".JPG"),
            os.path.join(self.image_dir, image_name + ".JPEG"),
            os.path.join(self.image_dir, image_name + ".PNG"),
            os.path.join(self.image_dir, image_name + ".webp"),
        ]

        image_path = None
        for p in possible_paths:
            if os.path.exists(p):
                image_path = p
                break

        if image_path is None:
            raise FileNotFoundError(f"Image not found for: {image_name}")

        image = Image.open(image_path).convert("RGB")
        label = int(row["target"])

        if self.transform:
            image = self.transform(image)

        return image, label


# =========================
# 5. MODEL
# =========================
class SkinCancerDenseNet(nn.Module):
    def __init__(self, num_classes=2, pretrained=True, freeze_features=True):
        super().__init__()

        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
        self.model = models.densenet121(weights=weights)

        if freeze_features:
            for param in self.model.features.parameters():
                param.requires_grad = False

        in_features = self.model.classifier.in_features
        self.model.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.model(x)


def build_model():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SkinCancerDenseNet(
        num_classes=2,
        pretrained=True,
        freeze_features=True
    ).to(device)
    return model, device


# =========================
# 6. TRANSFORMS
# =========================
def get_transforms(image_size=224):
    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    val_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    return train_transform, val_transform


# =========================
# 7. SUBSET WRAPPER
# =========================
class TransformSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]

        if self.transform:
            image = self.transform(image)

        return image, label


# =========================
# 8. DATALOADERS
# =========================
def create_dataloaders(csv_file, image_dir, batch_size=64, image_size=224, val_split=0.2):
    train_tf, val_tf = get_transforms(image_size)

    base_dataset = ISICBinaryDataset(csv_file, image_dir, transform=None)

    total_size = len(base_dataset)
    val_size = int(total_size * val_split)
    train_size = total_size - val_size

    train_subset, val_subset = random_split(base_dataset, [train_size, val_size])

    train_dataset = TransformSubset(train_subset, transform=train_tf)
    val_dataset = TransformSubset(val_subset, transform=val_tf)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    class_names = ["non-cancer", "cancer"]
    return train_loader, val_loader, class_names


# =========================
# 9. TRAINING LOOP
# =========================
def train_model(model, train_loader, val_loader, device, epochs=5, lr=1e-4):
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )

    scaler = torch.amp.GradScaler("cuda")
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0

    torch.backends.cudnn.benchmark = True

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")
        print("-" * 30)

        model.train()
        running_loss = 0.0
        running_corrects = 0
        total_train = 0

        for inputs, labels in tqdm.tqdm(train_loader):
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda"):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            _, preds = torch.max(outputs, 1)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels).item()
            total_train += labels.size(0)

        train_loss = running_loss / total_train
        train_acc = running_corrects / total_train

        model.eval()
        val_loss = 0.0
        val_corrects = 0
        total_val = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                with torch.amp.autocast("cuda"):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)

                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels).item()
                total_val += labels.size(0)

        epoch_val_loss = val_loss / total_val
        epoch_val_acc = val_corrects / total_val

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.4f}")

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_model_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_wts)
    return model


# =========================
# 10. RUN TRAINING
# =========================
model, device = build_model()
print("Using device:", device)

train_loader, val_loader, class_names = create_dataloaders(
    csv_file=CSV_FILE,
    image_dir=IMAGE_DIR,
    batch_size=64,
    image_size=224,
    val_split=0.2
)

trained_model = train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=5,
    lr=1e-4
)

torch.save({
    "model_state_dict": trained_model.state_dict(),
    "class_names": class_names,
    "image_size": 224
}, MODEL_SAVE_PATH)

print(f"Model saved to {MODEL_SAVE_PATH}")

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 121MB/s]


Using device: cuda
Rows after filtering: 9885
Class counts:
diagnosis_1
benign       8061
malignant    1824
Name: count, dtype: int64

Epoch 1/5
------------------------------


100%|██████████| 124/124 [10:39<00:00,  5.16s/it]


Train Loss: 0.5089 | Train Acc: 0.7922
Val Loss:   0.4755 | Val Acc:   0.8048

Epoch 2/5
------------------------------


...

Train Loss: 0.3990 | Train Acc: 0.8216
Val Loss:   0.3968 | Val Acc:   0.8098

Epoch 4/5
------------------------------


100%|██████████| 124/124 [01:05<00:00,  1.88it/s]


Train Loss: 0.3769 | Train Acc: 0.8244
Val Loss:   0.3743 | Val Acc:   0.8209

Epoch 5/5
------------------------------


100%|██████████| 124/124 [01:07<00:00,  1.82it/s]


Train Loss: 0.3630 | Train Acc: 0.8318
Val Loss:   0.3625 | Val Acc:   0.8316
Model saved to /content/drive/MyDrive/ISIC/models/densenet_skin_cancer_colab.pth
